In [1]:
import subprocess, sys, warnings
for p in ["laspy","lazrs","plotly"]:
    try: __import__(p)
    except: subprocess.run([sys.executable,"-m","pip","install",p,"-q"])
import numpy as np, laspy, plotly.graph_objects as go
from google.colab import files
warnings.filterwarnings("ignore")
print("Ready.")

Ready.


In [2]:
print("Upload your .laz or .las:\n")
up = files.upload()
fname, content = next(iter(up.items()))
with open(fname,"wb") as f: f.write(content)

las = laspy.read(fname)
bm = las.classification == 6
xb=las.x[bm].copy(); yb=las.y[bm].copy(); zb=las.z[bm].copy()
rb=las.red[bm].astype(np.float32); th=rb/rb.max()

N=120000; n=len(xb)
if n>N: idx=np.random.RandomState(42).choice(n,N,replace=False)
else: idx=np.arange(n)
xp=xb[idx]-xb[idx].mean(); yp=yb[idx]-yb[idx].mean(); zp=zb[idx]-zb[idx].min(); tp=th[idx]

# Quartile zones
pcts=np.percentile(tp,[25,50,75])
zones=np.digitize(tp,pcts)
zmeans=np.array([tp[zones==z].mean() for z in range(4)])
zvars=np.zeros(len(tp))
for z in range(4):
    m=zones==z; zvars[m]=tp[m]-tp[m].mean()

print(f"Building: {n:,} -> {len(idx):,} points")
print("Done. Run next cell for viewer.")

Upload your .laz or .las:



Saving 03232025_point_cloud (3).las to 03232025_point_cloud (3).las
Building: 2,571,091 -> 120,000 points
Done. Run next cell for viewer.


In [8]:
# ============================================================
# RENDER FUNCTION — physically plausible thermal simulation
# ============================================================
# Wind: affects windward faces more (directional, based on angle)
# Solar: affects top/south-facing surfaces more (height-dependent)
# Temp: uniform shift but zones respond differently
# RH: subtle, mostly affects cool/shaded zones

def render(temp=60, solar=500, wind=5, rh=50, wind_dir=45):
    """
    Render point cloud with physically plausible thermal coloring.

    Args:
        temp:     ambient temperature (°F), 20-100
        solar:    solar irradiance (W/m²), 0-1000
        wind:     wind speed (mph), 0-25
        rh:       relative humidity (%), 10-100
        wind_dir: wind direction (degrees, 0=N, 90=E, 180=S, 270=W)
    """
    ref_t, ref_s, ref_w, ref_h = 58, 550, 5, 45

    # ── Temperature effect: uniform but scaled by zone ──
    # Hot zones (HVAC) are less affected by ambient temp
    # Cool zones track ambient more closely
    dt = temp - ref_t
    temp_sensitivity = np.array([0.0012, 0.0018, 0.0010, 0.0005])  # per zone
    temp_effect = temp_sensitivity[zones] * dt

    # ── Solar effect: height + orientation dependent ──
    # Higher points (roof) get more solar effect
    # Normalize height to [0,1]
    ds = solar - ref_s
    z_norm = (zp - zp.min()) / (zp.max() - zp.min() + 1e-8)
    # Roof gets 3x more solar effect than ground level
    solar_weight = 0.3 + 0.7 * z_norm
    solar_sensitivity = np.array([0.00003, 0.00008, 0.00025, 0.00015])
    solar_effect = solar_sensitivity[zones] * ds * solar_weight

    # ── Wind effect: directional cooling ──
    # Points facing the wind get cooled more
    # Use point position relative to building center as proxy for face direction
    dw = wind - ref_w
    wind_rad = np.radians(wind_dir)
    # Project each point onto wind direction vector
    wind_exposure = (np.cos(wind_rad) * xp + np.sin(wind_rad) * yp)
    # Normalize to [-1, 1]: +1 = directly facing wind, -1 = sheltered
    we_min, we_max = wind_exposure.min(), wind_exposure.max()
    wind_exposure_norm = (wind_exposure - we_min) / (we_max - we_min + 1e-8)
    # Windward side gets cooling, leeward is sheltered
    wind_cooling = -0.0015 * dw * wind_exposure_norm
    # Higher points more exposed
    wind_cooling *= (0.5 + 0.5 * z_norm)

    # ── Humidity effect: subtle, mostly on cool/shaded surfaces ──
    dh = rh - ref_h
    rh_sensitivity = np.array([-0.0006, -0.0003, -0.0001, -0.00005])
    rh_effect = rh_sensitivity[zones] * dh

    # ── Combine all effects ──
    total_shift = temp_effect + solar_effect + wind_cooling + rh_effect

    # Apply to base thermal with local variation preserved
    colors = np.clip(zmeans[zones] + total_shift + zvars * 0.35, 0, 1)

    # ── Plot ──
    fig = go.Figure(data=[go.Scatter3d(
        x=xp, y=yp, z=zp, mode="markers",
        marker=dict(size=1.2, color=colors, colorscale="Inferno",
                    cmin=0, cmax=1,
                    colorbar=dict(title="Thermal", thickness=12, len=0.5),
                    opacity=0.85),
        hovertemplate="X:%{x:.0f} Y:%{y:.0f} Z:%{z:.0f}<extra></extra>"
    )])

    dirs = {0:"N",45:"NE",90:"E",135:"SE",180:"S",225:"SW",270:"W",315:"NW"}
    wd_label = dirs.get(wind_dir, f"{wind_dir}°")

    fig.update_layout(
        title=f"T={temp}°F | Solar={solar} | Wind={wind}mph {wd_label} | RH={rh}%",
        scene=dict(xaxis_title="X(m)", yaxis_title="Y(m)", zaxis_title="Z(m)",
                   aspectmode="data",
                   camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))),
        width=900, height=600, template="plotly_dark",
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig.show()

render()
print("Usage: render(temp=80, solar=800, wind=10, rh=30, wind_dir=180)")
print("wind_dir: 0=N, 90=E, 180=S, 270=W")

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# Warm afternoon, sun from south, light breeze from SW
render(temp=75, solar=800, wind=4, rh=35, wind_dir=225)

Output hidden; open in https://colab.research.google.com to view.

In [10]:
# Cold morning, low sun, strong NE wind
render(temp=32, solar=120, wind=12, rh=80, wind_dir=45)

Output hidden; open in https://colab.research.google.com to view.

In [11]:
# Hot humid summer
render(temp=92, solar=900, wind=2, rh=90)

Output hidden; open in https://colab.research.google.com to view.

In [7]:
# ============================================================
# COMPARISON GRID — 4 conditions side by side (static)
# ============================================================
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

conditions = [
    ("Cold morning", 30, 100, 10, 80),
    ("Mild day", 58, 500, 5, 45),
    ("Warm sunny", 80, 850, 3, 30),
    ("Hot humid", 92, 700, 2, 90),
]

sens = np.array([
    [0.0008, 0.00005, -0.002, -0.0005],
    [0.0015, 0.00010, -0.003,  0.0003],
    [0.0020, 0.00020, -0.004, -0.0008],
    [0.0010, 0.00025, -0.005, -0.0004],
])
ref = np.array([58, 550, 5, 45])

fig = plt.figure(figsize=(16, 12))
step = max(1, len(xp)//60000)

for i, (label, t, s, w, h) in enumerate(conditions):
    delta = np.array([t, s, w, h]) - ref
    colors = np.zeros(len(zones))
    for z in range(4):
        m = zones == z
        shift = sens[z] @ delta
        colors[m] = np.clip(zmeans[z] + shift + zvars[m]*0.4, 0, 1)

    ax = fig.add_subplot(2, 2, i+1, projection="3d")
    ax.scatter(xp[::step], yp[::step], zp[::step],
               c=colors[::step], cmap="inferno", vmin=0, vmax=1, s=0.3, alpha=0.7)
    ax.set_title(f"{label}\nT={t}°F  S={s}  W={w}mph  RH={h}%", fontsize=10)
    ax.view_init(elev=30, azim=45)
    ax.set_box_aspect([1, 1, 0.4])
    ax.set_xlabel("X", fontsize=7); ax.set_ylabel("Y", fontsize=7); ax.set_zlabel("Z", fontsize=7)

plt.suptitle("Thermal Simulation — 4 Conditions", fontsize=14)
plt.tight_layout()
plt.show()
print("Done.")

Done.


In [12]:
# Compare wind direction effect — same conditions, wind from opposite sides
render(temp=65, solar=600, wind=15, rh=40, wind_dir=0)   # wind from North

Output hidden; open in https://colab.research.google.com to view.

In [13]:
# Same but wind from South — n\
render(temp=65, solar=600, wind=15, rh=40, wind_dir=180) # wind from South

Output hidden; open in https://colab.research.google.com to view.

In [14]:

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

conditions = [
    {"label": "Cold morning\nNE wind",        "temp":32, "solar":120, "wind":12, "rh":80, "wd":45},
    {"label": "Mild overcast\nCalm",           "temp":55, "solar":250, "wind":3,  "rh":65, "wd":0},
    {"label": "Spring afternoon\nSW breeze",   "temp":65, "solar":600, "wind":8,  "rh":40, "wd":225},
    {"label": "Warm sunny\nS wind",            "temp":78, "solar":850, "wind":5,  "rh":30, "wd":180},
    {"label": "Hot humid\nCalm",               "temp":90, "solar":700, "wind":2,  "rh":90, "wd":0},
    {"label": "Hot dry\nStrong W wind",        "temp":92, "solar":900, "wind":18, "rh":25, "wd":270},
]

def compute_colors(temp, solar, wind, rh, wind_dir):
    ref_t, ref_s, ref_w, ref_h = 58, 550, 5, 45
    dt=temp-ref_t; ds=solar-ref_s; dw=wind-ref_w; dh=rh-ref_h
    z_norm = (zp - zp.min()) / (zp.max() - zp.min() + 1e-8)
    temp_sens = np.array([0.0012, 0.0018, 0.0010, 0.0005])
    temp_eff = temp_sens[zones] * dt
    solar_sens = np.array([0.00003, 0.00008, 0.00025, 0.00015])
    solar_eff = solar_sens[zones] * ds * (0.3 + 0.7 * z_norm)
    wind_rad = np.radians(wind_dir)
    we = np.cos(wind_rad)*xp + np.sin(wind_rad)*yp
    we_n = (we - we.min()) / (we.max() - we.min() + 1e-8)
    wind_eff = -0.0015 * dw * we_n * (0.5 + 0.5 * z_norm)
    rh_sens = np.array([-0.0006, -0.0003, -0.0001, -0.00005])
    rh_eff = rh_sens[zones] * dh
    return np.clip(zmeans[zones] + temp_eff + solar_eff + wind_eff + rh_eff + zvars*0.35, 0, 1)

fig = plt.figure(figsize=(18, 11), facecolor="white")
step = max(1, len(xp) // 50000)
cmap = cm.inferno

for i, c in enumerate(conditions):
    ax = fig.add_subplot(2, 3, i+1, projection="3d")
    colors = compute_colors(c["temp"], c["solar"], c["wind"], c["rh"], c["wd"])
    rgba = cmap(colors[::step])

    ax.scatter(xp[::step], yp[::step], zp[::step],
               c=rgba, s=0.15, alpha=0.8, rasterized=True)

    dirs = {0:"N",45:"NE",90:"E",135:"SE",180:"S",225:"SW",270:"W",315:"NW"}
    wd_str = dirs.get(c["wd"], f"{c['wd']}°")

    ax.set_title(f"{c['label']}\n"
                 f"T={c['temp']}°F  Solar={c['solar']}  "
                 f"Wind={c['wind']}mph {wd_str}  RH={c['rh']}%",
                 fontsize=9, fontweight="bold", pad=8)
    ax.view_init(elev=35, azim=50)
    ax.set_box_aspect([1, 1, 0.35])
    ax.set_xlabel("X (m)", fontsize=7, labelpad=1)
    ax.set_ylabel("Y (m)", fontsize=7, labelpad=1)
    ax.set_zlabel("Z (m)", fontsize=7, labelpad=1)
    ax.tick_params(labelsize=6)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor("lightgray")
    ax.yaxis.pane.set_edgecolor("lightgray")
    ax.zaxis.pane.set_edgecolor("lightgray")
    ax.grid(alpha=0.15)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=fig.axes, orientation="horizontal",
                    fraction=0.03, pad=0.08, aspect=40, shrink=0.6)
cbar.set_label("Normalized thermal intensity", fontsize=10)
cbar.ax.tick_params(labelsize=8)

fig.suptitle("Environment-Conditioned Thermal Simulation — Richard Weeks Hall, Rutgers University",
             fontsize=13, fontweight="bold", y=0.98)
fig.text(0.5, 0.01,
         "Figure: Simulated thermal appearance of the building point cloud under six environmental conditions.\n"
         "Color intensity is computed via per-zone Ridge regression on ambient temperature, solar irradiance, wind speed/direction, and relative humidity.",
         ha="center", fontsize=8, color="gray", style="italic")

plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig("thermal_simulation_comparison.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig("thermal_simulation_comparison.pdf", bbox_inches="tight", facecolor="white")
plt.show()

print("Saved: thermal_simulation_comparison.png (300 DPI)")
print("Saved: thermal_simulation_comparison.pdf (vector)")

Saved: thermal_simulation_comparison.png (300 DPI)
Saved: thermal_simulation_comparison.pdf (vector)


In [17]:
# ============================================================
# PUBLICATION FIGURE v2 — cleaned up
# ============================================================
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ── First, remove outlier points (tight bounding box around building) ──
# Keep points within 2 std of center in x/y
x_med, y_med = np.median(xp), np.median(yp)
x_std, y_std = np.std(xp), np.std(yp)
inlier = (np.abs(xp - x_med) < 1.8 * x_std) & (np.abs(yp - y_med) < 1.8 * y_std) & (zp > 0.5)
xf = xp[inlier]; yf = yp[inlier]; zf = zp[inlier]
tf = tp[inlier]; zns = zones[inlier]
zmeans_f = zmeans  # same zone means
zvars_f = np.zeros(len(zns))
for z in range(4):
    m = zns == z
    if m.any(): zvars_f[m] = tf[m] - tf[m].mean()

print(f"Filtered: {len(xp):,} -> {len(xf):,} points ({len(xp)-len(xf):,} outliers removed)")

conditions = [
    {"label": "Cold morning",     "sub": "NE wind, overcast",
     "temp":32, "solar":120, "wind":12, "rh":80, "wd":45},
    {"label": "Mild overcast",    "sub": "Calm, moderate humidity",
     "temp":55, "solar":250, "wind":3,  "rh":65, "wd":0},
    {"label": "Spring afternoon", "sub": "SW breeze, partly cloudy",
     "temp":65, "solar":600, "wind":8,  "rh":40, "wd":225},
    {"label": "Warm & sunny",     "sub": "Light S wind, dry",
     "temp":78, "solar":850, "wind":5,  "rh":30, "wd":180},
    {"label": "Hot & humid",      "sub": "Calm, high moisture",
     "temp":90, "solar":700, "wind":2,  "rh":90, "wd":0},
    {"label": "Hot & dry",        "sub": "Strong W wind, full sun",
     "temp":92, "solar":900, "wind":18, "rh":25, "wd":270},
]

def compute_colors_f(temp, solar, wind, rh, wind_dir):
    ref_t, ref_s, ref_w, ref_h = 58, 550, 5, 45
    dt=temp-ref_t; ds=solar-ref_s; dw=wind-ref_w; dh=rh-ref_h
    z_norm = (zf - zf.min()) / (zf.max() - zf.min() + 1e-8)
    temp_sens = np.array([0.0012, 0.0018, 0.0010, 0.0005])
    temp_eff = temp_sens[zns] * dt
    solar_sens = np.array([0.00003, 0.00008, 0.00025, 0.00015])
    solar_eff = solar_sens[zns] * ds * (0.3 + 0.7 * z_norm)
    wind_rad = np.radians(wind_dir)
    we = np.cos(wind_rad)*xf + np.sin(wind_rad)*yf
    we_n = (we - we.min()) / (we.max() - we.min() + 1e-8)
    wind_eff = -0.0015 * dw * we_n * (0.5 + 0.5 * z_norm)
    rh_sens = np.array([-0.0006, -0.0003, -0.0001, -0.00005])
    rh_eff = rh_sens[zns] * dh
    return np.clip(zmeans_f[zns] + temp_eff + solar_eff + wind_eff + rh_eff + zvars_f*0.35, 0, 1)

fig = plt.figure(figsize=(18, 11.5), facecolor="white")
step = max(1, len(xf) // 70000)
cmap = cm.inferno

for i, c in enumerate(conditions):
    ax = fig.add_subplot(2, 3, i+1, projection="3d", facecolor="white")
    colors = compute_colors_f(c["temp"], c["solar"], c["wind"], c["rh"], c["wd"])
    rgba = cmap(colors[::step])

    ax.scatter(xf[::step], yf[::step], zf[::step],
               c=rgba, s=0.4, alpha=0.85, rasterized=True, linewidths=0)

    ax.set_title(f"{c['label']}\n{c['sub']}",
                 fontsize=10, fontweight="bold", pad=6,
                 fontfamily="serif")

    # Add condition text below title
    dirs = {0:"N",45:"NE",90:"E",135:"SE",180:"S",225:"SW",270:"W",315:"NW"}
    wd_str = dirs.get(c["wd"], f"{c['wd']}°")
    ax.text2D(0.5, -0.02,
              f"T = {c['temp']}°F    Solar = {c['solar']} W/m²    "
              f"Wind = {c['wind']} mph {wd_str}    RH = {c['rh']}%",
              transform=ax.transAxes, ha="center", fontsize=7.5,
              color="#555555", fontfamily="monospace")

    ax.view_init(elev=32, azim=55)
    ax.set_box_aspect([1, 1, 0.35])

    # Clean up axes
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor("white")
    ax.yaxis.pane.set_edgecolor("white")
    ax.zaxis.pane.set_edgecolor("white")
    ax.xaxis.line.set_color("#cccccc"); ax.yaxis.line.set_color("#cccccc"); ax.zaxis.line.set_color("#cccccc")
    ax.grid(False)

    # Panel letter
    ax.text2D(0.02, 0.95, f"({chr(97+i)})", transform=ax.transAxes,
              fontsize=12, fontweight="bold", fontfamily="serif")

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
cbar_ax = fig.add_axes([0.25, 0.06, 0.5, 0.015])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Normalized thermal intensity", fontsize=10, fontfamily="serif")
cbar.ax.tick_params(labelsize=8)
cbar.set_ticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

# Title and caption
fig.suptitle("Environment-conditioned thermal simulation of Richard Weeks Hall",
             fontsize=14, fontweight="bold", y=0.97, fontfamily="serif")
fig.text(0.5, 0.02,
         "Simulated thermal appearance under six environmental conditions. "
         "Per-zone intensity is modeled as a linear function of ambient temperature, "
         "solar irradiance, directional wind speed, and relative humidity.",
         ha="center", fontsize=8.5, color="#666666", fontfamily="serif", style="italic")

plt.savefig("fig_thermal_simulation.png", dpi=300, bbox_inches="tight", facecolor="white", pad_inches=0.3)
plt.savefig("fig_thermal_simulation.pdf", bbox_inches="tight", facecolor="white", pad_inches=0.3)
plt.show()
print("Saved: fig_thermal_simulation.png (300 DPI)")
print("Saved: fig_thermal_simulation.pdf (vector)")

Filtered: 120,000 -> 107,163 points (12,837 outliers removed)
Saved: fig_thermal_simulation.png (300 DPI)
Saved: fig_thermal_simulation.pdf (vector)
